# HCP fMRI time-series and classical metric (ReHo, f/ALFF) extraction

This notebook demonstrates the timeseries extraction workflow for HCP dataset. It extracts atlas-wise time series and optionally computes ReHo, ALFF, and fALFF using the parameters in `configs/hcp_timeseries_parameters.jsonc`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if not (PROJECT_ROOT / "configs").is_dir():
    raise FileNotFoundError("Run this notebook from the RISE repository root or pipeline directory")
from rise import (
    build_hcp_run_inputs,
    extract_dataset,
    load_config,
    prepare_atlases,
    read_subject_ids,
)

### Configuration

Relative paths in the JSONC file are resolved from the configuration directory. Edit the file for persistent defaults, or pass an `overrides` dictionary for one run.

In [ ]:
CONFIG_FILE = PROJECT_ROOT / "configs" / "hcp_timeseries_parameters.jsonc"
config = load_config(CONFIG_FILE)
config


### Discover HCP runs

The training and test subject lists bundled under `data/HCP/info` are combined here.

In [ ]:
subjects = read_subject_ids(*config["subject_files"])
run_inputs = build_hcp_run_inputs(subjects, config)
print(f"Found {len(run_inputs)} runs for {len(subjects)} subjects.")
run_inputs[:2]

### Prepare atlas files

The packaged atlases and brain mask are resampled to match the resolution of the first available HCP run and stored in the configured cache. If the target data resolution matches one of the pre-built versions (1 mm, 2 mm, or 3 mm) provided in atlas_repo, that version should be used directly to avoid interpolation — resampling is only performed when no matching resolution is available.

In [ ]:
if not run_inputs:
    raise RuntimeError("No HCP runs were found. Check dataset_root and subject IDs.")

atlas_cache = prepare_atlases(run_inputs[0]["nii_file"], config, force=False)
print(f"Prepared atlases: {atlas_cache}")

### Run extraction

Outputs are written under `processed_data/indiv/<subject>/func/<run>/data`.

In [ ]:
run_inputs

In [ ]:
results = extract_dataset(
    run_inputs[:8],#Only the first 8 runs were executed as an example
    config,
    n_processes=config["n_processes"],
)
results